In [1]:
import sys
import asyncio
import time
import os

import numpy as np

from lsst.ts import salobj

from lsst.ts.observatory.control.auxtel.atcs import ATCS
from lsst.ts.observatory.control.auxtel.latiss import LATISS

In [2]:
print(os.environ["OSPL_URI"])
print(os.environ["LSST_DDS_PARTITION_PREFIX"])

file:///home/craiglagegit/WORK/ts_ddsconfig/config/ospl-sp.xml
summit


In [3]:
import logging
stream_handler = logging.StreamHandler(sys.stdout)
logger = logging.getLogger()
logger.addHandler(stream_handler)
logger.level = logging.DEBUG

In [4]:
#get classes and start them
domain = salobj.Domain()
await asyncio.sleep(10) # This can be removed in the future...
atcs = ATCS(domain)
latiss = LATISS(domain)
await asyncio.gather(atcs.start_task, latiss.start_task)

atmcs: Adding all resources.
atptg: Adding all resources.
ataos: Adding all resources.
atpneumatics: Adding all resources.
athexapod: Adding all resources.
atdome: Adding all resources.
atdometrajectory: Adding all resources.
atcamera: Adding all resources.
atspectrograph: Adding all resources.
atheaderservice: Adding all resources.
atarchiver: Adding all resources.
Read historical data in 0.09 sec
Read historical data in 0.11 sec
Read 1 history items for RemoteEvent(ATDomeTrajectory, 0, algorithm)
Read 2 history items for RemoteEvent(ATDomeTrajectory, 0, appliedSettingsMatchStart)
Read 1 history items for RemoteEvent(ATDomeTrajectory, 0, authList)
Read 100 history items for RemoteEvent(ATDomeTrajectory, 0, heartbeat)
Read 1 history items for RemoteEvent(ATDomeTrajectory, 0, logLevel)
Read 4 history items for RemoteEvent(ATDomeTrajectory, 0, logMessage)
Read 1 history items for RemoteEvent(ATDomeTrajectory, 0, settingVersions)
Read 1 history items for RemoteEvent(ATDomeTrajectory, 0, s

[[None, None, None, None, None, None, None], [None, None, None, None]]

mountStatus DDS read queue is filling: 94 of 100 elements
guidingAndOffsets DDS read queue is filling: 94 of 100 elements
currentTargetStatus DDS read queue is filling: 94 of 100 elements


In [ ]:
tmp = atcs.rem.atptg.evt_summaryState.get()
print(salobj.State(tmp.summaryState))

In [ ]:
tmp = atcs.rem.atptg.evt_heartbeat.get()
print(tmp)

In [ ]:
tmp = await latiss.rem.atheaderservice.evt_heartbeat.next(flush=True)
print(tmp)

In [ ]:
tmp = latiss.rem.atarchiver.evt_heartbeat.get()
print(tmp)

In [5]:
# enable components
await atcs.enable()
#await latiss.enable()

Enabling all components
Gathering settings.
Couldn't get settingVersions event. Using empty settings.
Complete settings for atmcs.
Complete settings for atptg.
Complete settings for ataos.
Complete settings for atpneumatics.
Complete settings for athexapod.
Complete settings for atdome.
Complete settings for atdometrajectory.
Settings versions: {'atmcs': '                                                                                                                               ', 'atptg': '', 'ataos': 'current', 'atpneumatics': '                                                                                                                               ', 'athexapod': 'summit', 'atdome': 'test', 'atdometrajectory': ''}
[atmcs]::[<State.STANDBY: 5>, <State.DISABLED: 1>, <State.ENABLED: 2>]
[atptg]::[<State.ENABLED: 2>]
[ataos]::[<State.ENABLED: 2>]
[atpneumatics]::[<State.STANDBY: 5>, <State.DISABLED: 1>, <State.ENABLED: 2>]
[athexapod]::[<State.ENABLED: 2>]
[atdome]::[<State.STANDB

In [7]:
await salobj.set_summary_state(atcs.rem.atdometrajectory, salobj.State.STANDBY)

[<State.ENABLED: 2>, <State.DISABLED: 1>, <State.STANDBY: 5>]

In [6]:
await salobj.set_summary_state(atcs.rem.atdome, salobj.State.STANDBY, settingsToApply='test')

[<State.ENABLED: 2>, <State.DISABLED: 1>, <State.STANDBY: 5>]

In [8]:
# take event checking out the slew commands to test telescope only
atcs.check.atdome = False
atcs.check.atdometrajectory = False

In [9]:
# Need to manually say which rotator we're using at the moment (N2-->focus=3, N1-->focus=2)
await atcs.rem.atptg.cmd_focusName.set_start(focus=3)

In [10]:
await salobj.set_summary_state(atcs.rem.atmcs, salobj.State.ENABLED)

[<State.ENABLED: 2>]

In [11]:
tmp = await atcs.rem.ataos.cmd_enableCorrection.set_start(m1=True, hexapod=True, atspectrograph=True)

In [12]:
start_az=-70
start_el=51
start_rot_pa=1
await atcs.point_azel(start_az, start_el, rot_tel=start_rot_pa, wait_dome=False)

Sending command
Stop tracking.
Scheduling check coroutines
process as completed...
atmcs: <State.ENABLED: 2>
atptg: <State.ENABLED: 2>
ataos: <State.ENABLED: 2>
atpneumatics: <State.ENABLED: 2>
athexapod: <State.ENABLED: 2>
Got False
Telescope not in position
[Telescope] delta Alt = +000.689 deg; delta Az= -000.000 deg; delta N1 = -001.003 deg; delta N2 = -000.024 deg 
[Telescope] delta Alt = +000.654 deg; delta Az= +000.000 deg; delta N1 = -000.343 deg; delta N2 = +000.000 deg 
[Telescope] delta Alt = -000.000 deg; delta Az= +000.033 deg; delta N1 = +000.001 deg; delta N2 = -000.000 deg 
Got True
Waiting for telescope to settle.
[Telescope] delta Alt = -000.000 deg; delta Az= -000.000 deg; delta N1 = +000.001 deg; delta N2 = -000.000 deg 
Telescope in position.


In [13]:
# get offsets from ATAOS
focus_offsets = atcs.rem.ataos.evt_focusOffsetSummary.get()
print(focus_offsets)

private_revCode: d8296953, private_sndStamp: 1610574234.4266376, private_rcvStamp: 1610732350.07064, private_seqNum: 51, private_identity: ATAOS, private_origin: 165593, private_host: 0, total: -0.02800000086426735, userApplied: 0.0, filter: -0.0020000000949949026, disperser: -0.026000000536441803, wavelength: 0.0, priority: 0


In [14]:
# get pointing offsets from ATAOS
pointing_offsets = atcs.rem.ataos.evt_pointingOffsetSummary.get()
print(pointing_offsets)

private_revCode: e1bb3fb9, private_sndStamp: 1610732843.5238767, private_rcvStamp: 1610732843.5246856, private_seqNum: 14, private_identity: ATAOS, private_origin: 165593, private_host: 0, total: [0.0, 0.0], filter: [0.0, 0.0], disperser: [0.0, 0.0], priority: 0


In [15]:
# How to slew to a target
await atcs.slew_object('HIP 90096')

Starting new HTTP connection (1): simbad.u-strasbg.fr:80
http://simbad.u-strasbg.fr:80 "POST /simbad/sim-script HTTP/1.1" 200 None
Slewing to HIP 90096: 18 23 12.1625 -12 00 53.117
Auto sky angle: 0.0 deg
Sending command
Stop tracking.
target python read queue is filling: 99 of 100 elements
mount_AzEl_Encoders python read queue is filling: 18 of 100 elements
Scheduling check coroutines
process as completed...
atmcs: <State.ENABLED: 2>
atptg: <State.ENABLED: 2>
ataos: <State.ENABLED: 2>
atpneumatics: <State.ENABLED: 2>
athexapod: <State.ENABLED: 2>
Got False
Telescope not in position
[Telescope] delta Alt = -002.353 deg; delta Az= -002.639 deg; delta N1 = +000.000 deg; delta N2 = +007.654 deg 
[Telescope] delta Alt = -002.254 deg; delta Az= -002.586 deg; delta N1 = -000.000 deg; delta N2 = +004.259 deg 
[Telescope] delta Alt = -000.019 deg; delta Az= -000.080 deg; delta N1 = -000.000 deg; delta N2 = +000.086 deg 
Got True
Waiting for telescope to settle.
[Telescope] delta Alt = -000.003

In [16]:
# get pointing offsets from ATAOS
pointing_offsets = atcs.rem.ataos.evt_pointingOffsetSummary.get()
print(pointing_offsets)

private_revCode: e1bb3fb9, private_sndStamp: 1610732843.5238767, private_rcvStamp: 1610732843.5246856, private_seqNum: 14, private_identity: ATAOS, private_origin: 165593, private_host: 0, total: [0.0, 0.0], filter: [0.0, 0.0], disperser: [0.0, 0.0], priority: 0


In [17]:
# change filter/grating using CSC directly
# Verify that ATAOS handles focus/pointing offsets from grating change
#await latiss.rem.atspectrograph.cmd_changeFilter.set_start(filter=1)
await latiss.rem.atspectrograph.cmd_changeDisperser.set_start(disperser=1)

In [18]:
# change filter/grating using CSC directly
# Verify that ATAOS handles focus/pointing offsets from grating change
await latiss.rem.atspectrograph.cmd_changeFilter.set_start(filter=1)
#await latiss.rem.atspectrograph.cmd_changeDisperser.set_start(disperser=1)

In [19]:
# reset offsets
tmp = await atcs.rem.ataos.cmd_resetOffset.set_start(axis='z')

In [20]:
# change filter/grating using CSC directly
# Verify that ATAOS handles focus/pointing offsets from grating change
await latiss.rem.atspectrograph.cmd_changeFilter.set_start(filter=2)
#await latiss.rem.atspectrograph.cmd_changeDisperser.set_start(disperser=1)

In [21]:
# change filter/grating using CSC directly
# Verify that ATAOS handles focus/pointing offsets from grating change
#await latiss.rem.atspectrograph.cmd_changeFilter.set_start(filter=2)
await latiss.rem.atspectrograph.cmd_changeDisperser.set_start(disperser=2)

In [22]:
# get pointing offsets from ATAOS
pointing_offsets = atcs.rem.ataos.evt_pointingOffsetSummary.get()
print(pointing_offsets)

private_revCode: e1bb3fb9, private_sndStamp: 1610734794.2657542, private_rcvStamp: 1610734794.2665431, private_seqNum: 15, private_identity: ATAOS, private_origin: 165593, private_host: 0, total: [20.0, 0.0], filter: [0.0, 0.0], disperser: [20.0, 0.0], priority: 0


In [23]:
# change filter/grating using CSC directly
# Verify that ATAOS handles focus/pointing offsets from grating change
await latiss.rem.atspectrograph.cmd_changeFilter.set_start(filter=1)
await latiss.rem.atspectrograph.cmd_changeDisperser.set_start(disperser=1)

In [24]:
# How to slew to a target
await atcs.slew_object('HIP 90004')

Resetting dropped connection: simbad.u-strasbg.fr
http://simbad.u-strasbg.fr:80 "POST /simbad/sim-script HTTP/1.1" 200 None
Slewing to HIP 90004: 18 21 49.7828 -11 55 21.653
Auto sky angle: 0.0 deg
Sending command
Stop tracking.
target python read queue is filling: 26 of 100 elements
Scheduling check coroutines
process as completed...
atmcs: <State.ENABLED: 2>
atptg: <State.ENABLED: 2>
ataos: <State.ENABLED: 2>
atpneumatics: <State.ENABLED: 2>
athexapod: <State.ENABLED: 2>
Got False
Telescope not in position
Got True
Waiting for telescope to settle.
Telescope in position.


In [25]:
# get pointing offsets from ATAOS
pointing_offsets = atcs.rem.ataos.evt_pointingOffsetSummary.get()
print(pointing_offsets)

private_revCode: e1bb3fb9, private_sndStamp: 1610735002.2609863, private_rcvStamp: 1610735002.261665, private_seqNum: 16, private_identity: ATAOS, private_origin: 165593, private_host: 0, total: [0.0, 0.0], filter: [0.0, 0.0], disperser: [0.0, 0.0], priority: 0


In [27]:
# change filter/grating using CSC directly
# Verify that ATAOS handles focus/pointing offsets from grating change
await latiss.rem.atspectrograph.cmd_changeFilter.set_start(filter=1)
await latiss.rem.atspectrograph.cmd_changeDisperser.set_start(disperser=2)

In [28]:
# get pointing offsets from ATAOS
pointing_offsets = atcs.rem.ataos.evt_pointingOffsetSummary.get()
print(pointing_offsets)

private_revCode: e1bb3fb9, private_sndStamp: 1610735690.7172644, private_rcvStamp: 1610735690.720298, private_seqNum: 17, private_identity: ATAOS, private_origin: 165593, private_host: 0, total: [20.0, 0.0], filter: [0.0, 0.0], disperser: [20.0, 0.0], priority: 0


In [29]:
# How to slew to a target
await atcs.slew_object('HIP 90096')

Slewing to HIP 90096: 18 23 12.1625 -12 00 53.117
Auto sky angle: 0.0 deg
Sending command
Stop tracking.
Scheduling check coroutines
process as completed...
atmcs: <State.ENABLED: 2>
atptg: <State.ENABLED: 2>
ataos: <State.ENABLED: 2>
atpneumatics: <State.ENABLED: 2>
athexapod: <State.ENABLED: 2>
allAxesInPosition python read queue is filling: 10 of 100 elements
Got False
Telescope not in position
Got True
Waiting for telescope to settle.
Telescope in position.


In [32]:
# get pointing offsets from ATAOS
pointing_offsets = await atcs.rem.ataos.evt_pointingOffsetSummary.aget()
print(pointing_offsets)

private_revCode: e1bb3fb9, private_sndStamp: 1610735690.7172644, private_rcvStamp: 1610735690.720298, private_seqNum: 17, private_identity: ATAOS, private_origin: 165593, private_host: 0, total: [20.0, 0.0], filter: [0.0, 0.0], disperser: [20.0, 0.0], priority: 0


/opt/lsst/software/stack/conda/miniconda3-py37_4.8.2/envs/lsst-scipipe-cb4e2dc/lib/python3.7/site-packages/ipykernel/__main__.py:9: RuntimeWarning: coroutine 'ReadTopic.aget' was never awaited


In [33]:
await atcs.stop_tracking()

Stop tracking.


ValueError: 9 is not a valid AtMountState

In [34]:
start_az=-70
start_el=51
start_rot_pa=1
await atcs.point_azel(start_az, start_el, rot_tel=start_rot_pa, wait_dome=False)

Sending command
Stop tracking.
Scheduling check coroutines
process as completed...
atmcs: <State.ENABLED: 2>
atptg: <State.ENABLED: 2>
ataos: <State.ENABLED: 2>
atpneumatics: <State.ENABLED: 2>
athexapod: <State.ENABLED: 2>
[Telescope] delta Alt = +008.731 deg; delta Az= +008.294 deg; delta N1 = +000.000 deg; delta N2 = -016.512 deg 
[Telescope] delta Alt = +008.601 deg; delta Az= +008.166 deg; delta N1 = +000.000 deg; delta N2 = -015.388 deg 
[Telescope] delta Alt = +006.447 deg; delta Az= +005.907 deg; delta N1 = +000.000 deg; delta N2 = -014.251 deg 
[Telescope] delta Alt = +002.606 deg; delta Az= +002.097 deg; delta N1 = +000.000 deg; delta N2 = -009.038 deg 
[Telescope] delta Alt = +000.001 deg; delta Az= +000.002 deg; delta N1 = +000.000 deg; delta N2 = -003.431 deg 
[Telescope] delta Alt = -000.000 deg; delta Az= -000.000 deg; delta N1 = -000.000 deg; delta N2 = -000.355 deg 
[Telescope] delta Alt = -000.000 deg; delta Az= -000.000 deg; delta N1 = -000.000 deg; delta N2 = +000.0

In [35]:
# get pointing offsets from ATAOS
pointing_offsets = await atcs.rem.ataos.evt_pointingOffsetSummary.aget()
print(pointing_offsets)

private_revCode: e1bb3fb9, private_sndStamp: 1610735690.7172644, private_rcvStamp: 1610735690.720298, private_seqNum: 17, private_identity: ATAOS, private_origin: 165593, private_host: 0, total: [20.0, 0.0], filter: [0.0, 0.0], disperser: [20.0, 0.0], priority: 0


In [37]:
await atcs.offset_xy( 60.0, 0.0, relative=True, persistent=True)

Calculating x/y offset: 60.0/0.0 
Applying Az/El offset: -38.56725632366335/-45.96266680323118 
Timed out waiting for offset done events.
Waiting for telescope to settle.
Done


In [38]:
await atcs.offset_azel( 60.0, 0.0, relative=True, persistent=True)

Applying Az/El offset: 60.0/0.0 
Persistent offset is not yet implemented (DM-21336).
Timed out waiting for offset done events.
Waiting for telescope to settle.
Done


In [39]:
await atcs.stop_tracking()

Stop tracking.


In [41]:
atcs.point_azel?

Signature:
atcs.point_azel(
    az,
    el,
    rot_tel=0.0,
    target_name='azel_target',
    wait_dome=False,
    slew_timeout=1200.0,
)
Docstring:
Slew the telescope to a fixed alt/az position.

Telescope will not track once it arrives in position.

Parameters
----------
az : `float`, `str` or astropy.coordinates.Angle
    Target Azimuth (degree). Accepts float (deg), sexagesimal string
    (DD:MM:SS.S or DD MM SS.S) coordinates or
    `astropy.coordinates.Angle`
el : `float` or `str`
    Target Elevation (degree). Accepts float (deg), sexagesimal string
    (DD:MM:SS.S or DD MM SS.S) coordinates or
    `astropy.coordinates.Angle`
rot_tel : `float` or `str`
    Specify rotator angle in mount physical coordinates.
    Accepts float (deg), sexagesimal string
    (DD:MM:SS.S or DD MM SS.S) coordinates or
    `astropy.coordinates.Angle`
target_name : `str`
    Name of the position.
wait_dome : `bool`
    Wait for dome to be in sync with the telescope? If preparing to
    take a flat, f

In [42]:
await atcs.point_azel(0.0, 80.0, rot_tel=0.0)

Sending command
Stop tracking.
Scheduling check coroutines
process as completed...
atmcs: <State.ENABLED: 2>
atptg: <State.ENABLED: 2>
ataos: <State.ENABLED: 2>
atpneumatics: <State.ENABLED: 2>
athexapod: <State.ENABLED: 2>
[Telescope] delta Alt = +028.961 deg; delta Az= +070.001 deg; delta N1 = +000.000 deg; delta N2 = -000.695 deg 
[Telescope] delta Alt = +028.900 deg; delta Az= +069.168 deg; delta N1 = +000.000 deg; delta N2 = +000.001 deg 
[Telescope] delta Alt = +028.723 deg; delta Az= +067.376 deg; delta N1 = +000.000 deg; delta N2 = -000.000 deg 
[Telescope] delta Alt = +027.059 deg; delta Az= +061.380 deg; delta N1 = +000.000 deg; delta N2 = -000.000 deg 
[Telescope] delta Alt = +024.858 deg; delta Az= +057.380 deg; delta N1 = +000.000 deg; delta N2 = -000.000 deg 
[Telescope] delta Alt = +020.095 deg; delta Az= +051.381 deg; delta N1 = +000.000 deg; delta N2 = +000.000 deg 
[Telescope] delta Alt = +016.281 deg; delta Az= +047.381 deg; delta N1 = -000.000 deg; delta N2 = +000.0

In [43]:
await salobj.set_summary_state(atcs.rem.ataos, salobj.State.STANDBY)

[<State.ENABLED: 2>, <State.DISABLED: 1>, <State.STANDBY: 5>]

In [44]:
await salobj.set_summary_state(atcs.rem.atpneumatics, salobj.State.STANDBY)

[<State.ENABLED: 2>, <State.DISABLED: 1>, <State.STANDBY: 5>]

In [45]:
await salobj.set_summary_state(atcs.rem.atpneumatics, salobj.State.ENABLED)

[<State.STANDBY: 5>, <State.DISABLED: 1>, <State.ENABLED: 2>]

In [46]:
await salobj.set_summary_state(atcs.rem.atpneumatics, salobj.State.STANDBY)

[<State.ENABLED: 2>, <State.DISABLED: 1>, <State.STANDBY: 5>]

In [48]:
await atcs.stop_tracking()

Stop tracking.


ValueError: 9 is not a valid AtMountState

In [49]:
await salobj.set_summary_state(atcs.rem.atmcs, salobj.State.STANDBY)

[<State.ENABLED: 2>, <State.DISABLED: 1>, <State.STANDBY: 5>]

In [50]:
await salobj.set_summary_state(atcs.rem.athexapod, salobj.State.STANDBY)

[<State.ENABLED: 2>, <State.DISABLED: 1>, <State.STANDBY: 5>]

In [ ]:
#await atcs.rem.atdome.cmd_homeAzimuth.start()

In [ ]:
tmp=await atcs.rem.athexapod.cmd_start.set_start(settingsToApply='summit')
print(tmp)

In [ ]:
tmp=await atcs.rem.athexapod.cmd_enable.start()
print(tmp)

In [ ]:
# Perform Rotator verification (LTS-337-003)True

In [ ]:
await attcs.stop_tracking()

In [ ]:
start_az=165.5
start_el=67.9
start_rot_pa=-170
await attcs.point_azel(start_az, start_el, rot_pa=start_rot_pa, wait_dome=True)

In [ ]:
start_az=165.5
start_el=67.9
start_rot_pa=170

start_time =time.time()
await attcs.point_azel(start_az, start_el, rot_pa=start_rot_pa, wait_dome=True)
end_time = time.time()
print(f'Time to perform {170*2} degree rotation of N2 is {end_time-start_time} seconds')

In [ ]:
# send telescope to alt/az/
#await salobj.set_summary_state(latiss.atcamera, salobj.State.ENABLED)

In [ ]:
# point telescope to desired starting position
# this make take extra time as more checks are performed
start_az=50
start_el=20
start_rot_pa=0
await attcs.point_azel(start_az, start_el, rot_pa=start_rot_pa, wait_dome=True)

In [ ]:
#declare offset size
d_az= 3.5 # degrees
d_el = 3.5 # degrees
d_rot= 3.5 # degrees

d_az= 10 # degrees
d_el = 10 # degrees
d_rot= 10 # degrees

d_az= 90 # degrees
d_el = 60 # degrees
d_rot= 90 # degrees

In [ ]:
from astropy.time import Time
from astropy.coordinates import AltAz, ICRS, EarthLocation, Angle, FK5
import astropy.units as u
location = EarthLocation.from_geodetic(lon=-70.747698*u.deg,
                                       lat=-30.244728*u.deg,
                                       height=2663.0*u.m)

In [ ]:
# get RA/DEC of current position
az = Angle(start_az, unit=u.deg)
el = Angle(start_el, unit=u.deg)
print(f'orig az {az} and el {el}')
time_data = await attcs.atptg.tel_timeAndDate.next(flush=True, timeout=2)
curr_time_atptg = Time(time_data.tai, format="mjd", scale="tai")
#print(curr_time_atptg)
coord_frame_AltAz = AltAz(location=location, obstime=curr_time_atptg)
coord_frame_radec = ICRS()
coord_azel = AltAz(az=az, alt=el, location=location, obstime=curr_time_atptg)
ra_dec = coord_azel.transform_to(coord_frame_radec)
print('Current Position is: \n {}'.format(coord_azel))
print('Current Position is: \n {}'.format(ra_dec))

# get RA/DEC of target position
az = Angle(start_az+d_az, unit=u.deg)
el = Angle(start_el+d_el, unit=u.deg)
print(f'target az {az} and el {el}')
coord_azel_target = AltAz(az=az, alt=el, location=location, obstime=curr_time_atptg)
ra_dec_target = coord_azel_target.transform_to(coord_frame_radec)
print('Target Position is: \n {}'.format(coord_azel_target))
print('Target Position is: \n {}'.format(ra_dec_target))


In [ ]:
# Why does doing no slew at all take ~32.7 sec?

#send to starting position
await attcs.slew_icrs(ra=str(ra_dec.ra), dec=str(ra_dec.dec), rot_pa=0.0, slew_timeout=240., stop_before_slew=True)


print('track for 2s')
await asyncio.sleep(2)
# take a quick image
await latiss.take_engtest(exptime=1)

print('Starting to Slew to target')
start_time = time.time()
await attcs.slew_icrs(ra=str(ra_dec_target.ra), dec=str(ra_dec_target.dec), rot_pa=d_rot, slew_timeout=240., stop_before_slew=False, wait_settle=False)
end_time = time.time()
print('Time to slew is {} [s]'.format(end_time-start_time))
await latiss.take_engtest(exptime=1)

In [ ]:
await attcs.stop_tracking()

In [ ]:
# Perform tracking test by taking continous 20s exposures
#await attcs.slew_object('HD 167060')
await attcs.slew_object('WD 1327-083')

In [ ]:
await latiss.take_object(exptime=15, grating='empty_1', filter='RG610')

In [ ]:
await latiss.take_object(exptime=15, grating='ronchi90lpmm', filter='empty_1')

In [ ]:
while True:
    await latiss.take_object(exptime=15, grating='empty_1', filter='RG610')
    await latiss.take_object(exptime=15, grating='ronchi90lpmm', filter='empty_1')

In [ ]:
# stopped tracking elsewhere, then reslewed
await attcs.slew_object('HD 59468')

In [ ]:
# OFFSET  AZ/EL
await attcs.offset_azel(az=-10*60*60, el=0)

In [ ]:
# again
await attcs.offset_azel(az=-20*60*60, el=0)

In [ ]:
# OFFSET  AZ/EL
await attcs.offset_azel(az=-10*60*60, el=0)

In [ ]:
# OFFSET  AZ/EL
await attcs.offset_azel(az=-5*60*60, el=0)
await attcs.offset_azel(az=-2*60*60, el=0)
await attcs.offset_azel(az=0, el=0)

In [ ]:
while True:
    await latiss.take_object(exptime=15, grating='empty_1', filter='RG610')
    await latiss.take_object(exptime=15, grating='ronchi90lpmm', filter='empty_1')
    

In [ ]:
await latiss.take_bias(nbias=1)

In [ ]:
#await attcs.startup()

In [ ]:
await latiss.enable()